# Pertemuan 9
Sholahuddin Ahmad (NPM: 250506070)  

### 1. Inisialisasi Koneksi MongoDB & Driver PyMongo

In [2]:
from pymongo import MongoClient

# Koneksi ke MongoDB lokal
client = MongoClient("mongodb://localhost:27017/")
db = client["praktikum_bigdata"]
koleksi_produk = db["produk"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database aktif:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database aktif: ['admin', 'config', 'local']


### 2. CRUD: Create & Read

In [7]:
# Bersihkan data lama
koleksi_produk.delete_many({})

# 1. insert_one
koleksi_produk.insert_one({
    "nama_produk": "Kopi Susu",
    "kategori": "Minuman",
    "harga": 18000,
    "stok": 120,
    "tags": ["best-seller", "dingin"]
})

# 2. insert_many
koleksi_produk.insert_many([
    {"nama_produk": "Roti Bakar", "kategori": "Makanan", "harga": 15000, "stok": 45, "tags": ["hangat"]},
    {"nama_produk": "Es Teh", "kategori": "Minuman", "harga": 8000, "stok": 200, "tags": ["dingin", "murah"]},
    {"nama_produk": "Kentang Goreng", "kategori": "Makanan", "harga": 20000, "stok": 60, "tags": ["gurih"]},
    {"nama_produk": "Cappuccino", "kategori": "Minuman", "harga": 25000, "stok": 80, "tags": ["hangat", "best-seller"]},
])

print("Jumlah dokumen di collection 'produk':", koleksi_produk.count_documents({}))

Jumlah dokumen di collection 'produk': 5


In [8]:
# Filter produk dengan harga di atas 15.000
print("Produk dengan harga > Rp15.000:")
for p in koleksi_produk.find({"harga": {"$gt": 15000}}):
    print(f"- {p['nama_produk']} (Rp{p['harga']:,})")

Produk dengan harga > Rp15.000:
- Kopi Susu (Rp18,000)
- Kentang Goreng (Rp20,000)
- Cappuccino (Rp25,000)


### 3. CRUD: Update & Delete

In [9]:
# update_one: Ganti harga Es Teh jadi 9000
koleksi_produk.update_one({"nama_produk": "Es Teh"}, {"$set": {"harga": 9000}})

# update_many: Simulasi penjualan, potong stok minuman sebanyak 5
koleksi_produk.update_many({"kategori": "Minuman"}, {"$inc": {"stok": -5}})

# delete_one: Hapus menu kentang goreng
koleksi_produk.delete_one({"nama_produk": "Kentang Goreng"})

DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)

### 4. Menjembatani MongoDB ke Apache Spark DataFrame

In [10]:
from spark_init import spark, F
import pandas as pd

# Langkah 1: Tarik seluruh data Mongo ke list of dict
data_mongo = list(koleksi_produk.find({}))

# Langkah 2: Ubah jadi Pandas DataFrame
df_p = pd.DataFrame(data_mongo)

# Langkah 3: Penjinakan tipe data BSON
# Wajib ubah ObjectId ke string murni biar JVM Spark nggak kaget
df_p["_id"] = df_p["_id"].astype(str)
# Format list array jadi string terpisah koma
df_p["tags"] = df_p["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")

# Langkah 4: Oper ke Spark DataFrame!
df_spark = spark.createDataFrame(df_p)

print("=== SKEMA SPARK DATAFRAME ===")
df_spark.printSchema()
df_spark.show(truncate=False)

# Agregasi Spark: Rata-rata harga & jumlah produk per kategori
df_spark.groupBy("kategori").agg(
    F.avg("harga").alias("rata_rata_harga"),
    F.count("nama_produk").alias("jumlah_produk")
).show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 23:20:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


⚡ SparkSession siap! (Spark 3.5.9 | local[*])
=== SKEMA SPARK DATAFRAME ===
root
 |-- _id: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- tags: string (nullable = true)



+------------------------+-----------+--------+-----+----+-------------------+
|_id                     |nama_produk|kategori|harga|stok|tags               |
+------------------------+-----------+--------+-----+----+-------------------+
|6aeb5a89e750e2f9f9f486a3|Kopi Susu  |Minuman |18000|115 |best-seller, dingin|
|6aeb5a89e750e2f9f9f486a4|Roti Bakar |Makanan |15000|45  |hangat             |
|6aeb5a89e750e2f9f9f486a5|Es Teh     |Minuman |9000 |195 |dingin, murah      |
|6aeb5a89e750e2f9f9f486a7|Cappuccino |Minuman |25000|75  |hangat, best-seller|
+------------------------+-----------+--------+-----+----+-------------------+



[Stage 2:>                                                          (0 + 2) / 2]

+--------+------------------+-------------+
|kategori|   rata_rata_harga|jumlah_produk|
+--------+------------------+-------------+
| Makanan|           15000.0|            1|
| Minuman|17333.333333333332|            3|
+--------+------------------+-------------+

